# PsychScanner Parser Tutorial

Walks through every parser mode currently supported by `psychscanner`:

| Mode | `parser=` value | What it does |
|------|-----------------|--------------|
| 1 | `"0"` | No structured parsing — raw `AIMessage` content |
| 2 | `"1"` | Read parser class name from the task JSON, resolve via `eval()` |
| 3 | `MyPydanticClass` | Pass a Pydantic class directly |
| 4 | `"dynamic"` | Route between two hardcoded RM parsers based on `trcode` |

Plus the cross-cutting flags `parser_raw` and `parser_config`.

## Backend requirement

Modes 2–4 call LangChain's `with_structured_output()`, which needs a real LLM provider (the bundled `mock-llm` does not support structured output). This notebook uses a local **Ollama** model — install Ollama and pull a small instruct model first:

```bash
ollama pull llama3.2:3b-instruct-fp16
```

If you prefer OpenAI/Anthropic/HuggingFace, swap the `MODEL_FAMILY` and `MODEL_NAME` in the setup cell.

## 1. Setup

In [6]:
import json
import shutil
from pathlib import Path
from pprint import pprint

import psychscanner as psy
from psychscanner import ExpCard, ExpCardInit, ScannerModel

# Backend — change these two lines if you don't use Ollama
MODEL_FAMILY = "ollama"
MODEL_NAME   = 'smollm2:360m-instruct-fp16'

PROJECT_DIR = Path.cwd() / "_parser_tutorial_runs"
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir()
print(f"Outputs will land in: {PROJECT_DIR}")

Outputs will land in: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_parser_tutorial_runs


## 2. What parsers ship with the package?

Two modules expose ready-made Pydantic parser classes:

- `psychscanner.datasets.prompts.parser` — survey rating + reality-monitoring (RM) parsers
- `psychscanner.datasets.prompts.parser_extra` — additional RM/task-specific parsers

Let's enumerate them:

In [7]:
import inspect
from pydantic import BaseModel
from psychscanner.datasets.prompts import parser as parser_mod
from psychscanner.datasets.prompts import parser_extra as parser_extra_mod

def list_parsers(mod):
    return [
        name for name, obj in inspect.getmembers(mod, inspect.isclass)
        if issubclass(obj, BaseModel) and obj is not BaseModel and obj.__module__ == mod.__name__
    ]

print("parser.py:")
for n in list_parsers(parser_mod): print(f"  - {n}")
print("\nparser_extra.py:")
for n in list_parsers(parser_extra_mod): print(f"  - {n}")

parser.py:
  - AllResponseRMEI
  - AllResponseRMEIN
  - AllResponseRMIE
  - AllResponseRMIEN
  - Confidence16
  - DefaultLiteralVivid010
  - DefaultLiteralVivid15
  - DefaultLiteralVivid15Pol
  - JudgmentEI
  - JudgmentEIN
  - JudgmentIE
  - JudgmentIEN
  - RelatednessRating
  - ResponseRmStEI
  - ResponseRmStEIN
  - ResponseRmStIE
  - ResponseRmStIEN
  - Response_part_1_rm
  - Response_part_2_rm
  - Response_part_2_rmrevo
  - TwoResponses
  - Word2

parser_extra.py:
  - DefaultLiteralAgree
  - DefaultParser
  - DefaultRMEncodingPhase
  - DefaultRatingParser
  - DefaultResponseRating
  - DefaultResponseRatingConvo
  - DefaultRmChoiceConf16
  - DefaultWordCaseNonWord
  - DefaultWordCaseNonWordConf16
  - Ready
  - ResponseRmScSt
  - SimpleResponseRating
  - Source
  - TaskReadyConfidence
  - TaskResponse
  - Task_1_ResponseRate
  - Task_2_ResponseRate
  - Task_3_ResponseRate
  - WordNonWord


Inspect a single parser to see what fields it expects:

In [8]:
from psychscanner.datasets.prompts.parser_extra import DefaultLiteralAgree

print("Schema:")
print(json.dumps(DefaultLiteralAgree.model_json_schema(), indent=2))

Schema:
{
  "description": "Give response on a Likert Scale of range 1 to 5 for the given item.",
  "properties": {
    "rating": {
      "description": "Agreement Rating.\n             Rate agreement on the scale of 1 to 5, where:\n             '5' to indicate that you absolutely agree that the statement describes you;\n             '1' to indicate that you totally disagree with the statement\n             '3' if you not sure, but always to make a choice.\n             Always give a single integer rating value ranging from 1 to 5 on the given item.",
      "enum": [
        1,
        2,
        3,
        4,
        5
      ],
      "title": "Rating",
      "type": "integer"
    }
  },
  "required": [
    "rating"
  ],
  "title": "DefaultLiteralAgree",
  "type": "object"
}


## 3. Mode 1 — `parser="0"` (no parsing)

The LLM response passes through unchanged. `with_structured_output()` is **not** called, so this mode works with **any** model — including the bundled `mock-llm`. We'll use that here so the cell runs without Ollama.

In [9]:
task_file = Path.cwd() / "tasks" / "example_survey.json"

card_no_parser = ExpCardInit()
card_no_parser.proj_dir    = PROJECT_DIR
card_no_parser.projectname = "mode1_no_parser"
card_no_parser.model       = "mock-chat-model"
card_no_parser.family      = "mock-llm"
card_no_parser.task_file   = task_file
card_no_parser.parser      = "0"           # <-- no parser
card_no_parser.cogtype     = "no"
card_no_parser.nsim        = 1
card_no_parser.tunnel_status = "0"

exp = ExpCard(card_no_parser)
scanner = ScannerModel(expcard=exp)
results_mode1 = scanner.run()

print("\n--- First trial response (raw, no parsing) ---")
first_trial = results_mode1[0][0]
print("pred_resp:", first_trial["pred_resp"])
print("type:    ", type(first_trial["pred_resp"]).__name__)

----<PROJECT AND DATA ROOT DIRECTORY>----
	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_parser_tutorial_runs
	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_parser_tutorial_runs/mode1_no_parser/example_survey/mock-llm_mock-chat-model_SingleTurn
----<>----
--<chat model>-- model_name='mock-chat-model' repeat_buffer_length=10


2026-05-03 03:44:07.222 | CRITICAL | psychscanner.session_tunnel.session_tunnel:create_tunnel:152 - BEGIN


TOTAL RUNS: 1	RESUME IDX: None
----<>---- task running


6it [00:00, 160.94it/s]
2026-05-03 03:44:07.340 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-05-03 03:44:07.347 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END



--- First trial response (raw, no parsing) ---
pred_resp: content='{\'content\': \'{\\n    "TRI\', \'additional_kwargs\': {}, \'response_metadata\': {\'time_in_seconds\': 3, \'model_name\': \'mock-chat-model\'}, \'type\': \'ai\', \'name\': None, \'id\': \'lc_run--019deccb-6a8b-71e1-af4b-6957b3b3ae36-0\', \'tool_calls\': [], \'invalid_tool_calls\': [], \'usage_metadata\': {\'input_tokens\': 333, \'output_tokens\': 10, \'total_tokens\': 343}}' additional_kwargs={} response_metadata={} id='39982bcd-a867-4e35-8437-64741ca97bed' tool_calls=[] invalid_tool_calls=[]
type:     AIMessage


Notice the response is an `AIMessage` whose `.content` is the raw model output. No JSON, no schema validation.

## 4. Mode 2 — `parser="1"` (resolve from task JSON)

The task JSON has a `"parser"` key holding the **class name as a string**. Setting `parser="1"` tells `ExpCard` to look that string up via `eval()` and use the resulting class.

Our [`tasks/example_survey.json`](tasks/example_survey.json) has `"parser": "DefaultLiteralAgree"`.

In [10]:
card_from_json = ExpCardInit()
card_from_json.proj_dir    = PROJECT_DIR
card_from_json.projectname = "mode2_from_json"
card_from_json.model       = MODEL_NAME
card_from_json.family      = MODEL_FAMILY
card_from_json.parameters  = {"temperature": 0}
card_from_json.task_file   = task_file
card_from_json.parser      = "1"           # <-- read parser name from task JSON
card_from_json.cogtype     = "no"
card_from_json.nsim        = 1
card_from_json.tunnel_status = "0"

exp = ExpCard(card_from_json)
print("Parser resolved to:", exp.parser.__name__)

scanner = ScannerModel(expcard=exp)
results_mode2 = scanner.run()

print("\n--- First trial response (parsed) ---")
pprint(results_mode2[0][0]["pred_resp"])

----<PROJECT AND DATA ROOT DIRECTORY>----
	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_parser_tutorial_runs
	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_parser_tutorial_runs/mode2_from_json/example_survey/ollama_smollm2:360m-instruct-fp16_SingleTurn
----<>----
Parser resolved to: DefaultLiteralAgree
{'temperature': 0}
--<chat model>-- model='smollm2:360m-instruct-fp16' temperature=0.0


2026-05-03 03:44:17.665 | CRITICAL | psychscanner.session_tunnel.session_tunnel:create_tunnel:152 - BEGIN


TOTAL RUNS: 1	RESUME IDX: None
----<>---- task running


6it [00:06,  1.01s/it]
2026-05-03 03:44:23.713 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-05-03 03:44:23.716 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END



--- First trial response (parsed) ---
AIMessage(content="{'rating': 5}", additional_kwargs={}, response_metadata={}, id='12b6eae4-d322-4b66-b15c-d7b1be46775a', tool_calls=[], invalid_tool_calls=[])


The response should now be a structured dict with a `rating` field constrained to `{1, 2, 3, 4, 5}`.

## 5. Mode 3 — pass a Pydantic class directly

Skip the JSON-name-and-eval round trip. This is the most explicit mode and the one to prefer when working in code.

In [11]:
from psychscanner.datasets.prompts.parser import DefaultLiteralVivid15

card_direct = ExpCardInit()
card_direct.proj_dir    = PROJECT_DIR
card_direct.projectname = "mode3_direct_class"
card_direct.model       = MODEL_NAME
card_direct.family      = MODEL_FAMILY
card_direct.parameters  = {"temperature": 0}
card_direct.task_file   = task_file
card_direct.parser      = DefaultLiteralVivid15  # <-- Pydantic class object
card_direct.cogtype     = "no"
card_direct.nsim        = 1
card_direct.tunnel_status = "0"

exp = ExpCard(card_direct)
scanner = ScannerModel(expcard=exp)
results_mode3 = scanner.run()

print("--- First trial response ---")
pprint(results_mode3[0][0]["pred_resp"])

----<PROJECT AND DATA ROOT DIRECTORY>----
	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_parser_tutorial_runs
	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_parser_tutorial_runs/mode3_direct_class/example_survey/ollama_smollm2:360m-instruct-fp16_SingleTurn
----<>----


{'temperature': 0}
--<chat model>-- model='smollm2:360m-instruct-fp16' temperature=0.0


2026-05-03 03:47:11.569 | CRITICAL | psychscanner.session_tunnel.session_tunnel:create_tunnel:152 - BEGIN


TOTAL RUNS: 1	RESUME IDX: None
----<>---- task running


6it [00:05,  1.06it/s]
2026-05-03 03:47:17.328 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-05-03 03:47:17.331 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


--- First trial response ---
AIMessage(content="{'Vividness': 1}", additional_kwargs={}, response_metadata={}, id='c7ea4671-ac9c-4954-9b4c-1f7210a07f1c', tool_calls=[], invalid_tool_calls=[])


### 5b. Custom parser of your own

Any `pydantic.BaseModel` subclass works. Here's a tiny one for a binary yes/no with a confidence rating.

In [12]:
from pydantic import BaseModel, Field
from typing import Literal

class YesNoConfidence(BaseModel):
    """Answer yes/no and rate your confidence on a 1–5 scale."""
    answer:     Literal["yes", "no"]      = Field(..., description="Your yes/no answer")
    confidence: Literal[1, 2, 3, 4, 5]    = Field(..., description="1=guess, 5=certain")

card_custom = ExpCardInit()
card_custom.proj_dir    = PROJECT_DIR
card_custom.projectname = "mode3_custom_class"
card_custom.model       = MODEL_NAME
card_custom.family      = MODEL_FAMILY
card_custom.parameters  = {"temperature": 0}
card_custom.task_file   = task_file
card_custom.parser      = YesNoConfidence  # <-- your custom class
card_custom.cogtype     = "no"
card_custom.nsim        = 1
card_custom.tunnel_status = "0"

exp = ExpCard(card_custom)
scanner = ScannerModel(expcard=exp)
results_custom = scanner.run()

pprint(results_custom[0][0]["pred_resp"])

----<PROJECT AND DATA ROOT DIRECTORY>----
	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_parser_tutorial_runs
	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_parser_tutorial_runs/mode3_custom_class/example_survey/ollama_smollm2:360m-instruct-fp16_SingleTurn
----<>----
{'temperature': 0}
--<chat model>-- model='smollm2:360m-instruct-fp16' temperature=0.0


2026-05-03 03:47:17.460 | CRITICAL | psychscanner.session_tunnel.session_tunnel:create_tunnel:152 - BEGIN


TOTAL RUNS: 1	RESUME IDX: None
----<>---- task running


6it [00:08,  1.38s/it]
2026-05-03 03:47:25.736 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-05-03 03:47:25.740 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


AIMessage(content="{'answer': 'yes', 'confidence': 1}", additional_kwargs={}, response_metadata={}, id='405ce948-962e-4b06-8f3c-2a1e258269fb', tool_calls=[], invalid_tool_calls=[])


## 6. Mode 4 — `parser="dynamic"` (route by `trcode`)

**This mode is hardcoded** for reality-monitoring tasks. In `psychscanner/memories/single_turn_convo.py`:

```python
def parser_selector(state):
    if "test" in state["trcode"]:
        return "runnable_resp2node"   # uses Response_part_2_rm
    return "runnable_resp1node"        # uses Response_part_1_rm
```

So the demo needs trials whose `trcode` contains `"test"` (test phase) and trials that don't (encoding phase). We'll build a minimal task on the fly:

In [13]:
rm_task = {
    "tasktype": "survey",
    "taskname": "rm_dynamic_demo",
    "instructions": {
        "definition": [
            "Encoding trials: report the second word and rate its relatedness 0–100.",
            "Test trials: judge whether the second word was internal/external and rate confidence 1–6."
        ]
    },
    "contexts": ["encode", "test"],
    "contexts_id": ["E", "T"],
    "context_present": True,
    "items": {
        "encode_1": [{"trcode": "encode_1", "stimulus": "Word 1: APPLE | Word 2: FRUIT"}],
        "encode_2": [{"trcode": "encode_2", "stimulus": "Word 1: TABLE | Word 2: ____ (imagine)"}],
        "test_1":   [{"trcode": "test_1",   "stimulus": "Word 1: APPLE | Word 2: FRUIT — judge."}],
        "test_2":   [{"trcode": "test_2",   "stimulus": "Word 1: TABLE | Word 2: CHAIR — judge."}]
    },
    "parser": "",
    "chain_type": "item"
}

rm_task_file = PROJECT_DIR / "rm_dynamic_demo.json"
rm_task_file.write_text(json.dumps(rm_task, indent=2))

card_dyn = ExpCardInit()
card_dyn.proj_dir    = PROJECT_DIR
card_dyn.projectname = "mode4_dynamic"
card_dyn.model       = MODEL_NAME
card_dyn.family      = MODEL_FAMILY
card_dyn.parameters  = {"temperature": 0}
card_dyn.task_file   = rm_task_file
card_dyn.parser      = "dynamic"            # <-- routes by trcode
card_dyn.cogtype     = "no"
card_dyn.nsim        = 1
card_dyn.tunnel_status = "0"

exp = ExpCard(card_dyn)
scanner = ScannerModel(expcard=exp)
results_dyn = scanner.run()

for trial in results_dyn[0]:
    print(f"trcode={trial['trcode']!r:12}  parsed={trial['pred_resp']}")

----<PROJECT AND DATA ROOT DIRECTORY>----
	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_parser_tutorial_runs
	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_parser_tutorial_runs/mode4_dynamic/rm_dynamic_demo/ollama_smollm2:360m-instruct-fp16_SingleTurn
----<>----


ValueError: 'encode' is not in list

Encoding trials should have `Word_2` + `Rating` fields. Test trials should have `Judgment` + `Confidence` fields. Different schema per trial, dispatched automatically by `trcode`.

## 7. `parser_raw=True` — keep the raw AIMessage alongside parsing

Useful for debugging when you want to see what the model emitted *before* Pydantic coerced it.

In [14]:
card_raw = ExpCardInit()
card_raw.proj_dir    = PROJECT_DIR
card_raw.projectname = "opt_parser_raw"
card_raw.model       = MODEL_NAME
card_raw.family      = MODEL_FAMILY
card_raw.parameters  = {"temperature": 0}
card_raw.task_file   = task_file
card_raw.parser      = DefaultLiteralVivid15
card_raw.parser_raw  = True                  # <-- keep raw alongside parsed
card_raw.cogtype     = "no"
card_raw.nsim        = 1
card_raw.tunnel_status = "0"

exp = ExpCard(card_raw)
scanner = ScannerModel(expcard=exp)
results_raw = scanner.run()

print("--- pred_resp with parser_raw=True ---")
pprint(results_raw[0][0]["pred_resp"])

----<PROJECT AND DATA ROOT DIRECTORY>----
	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_parser_tutorial_runs
	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_parser_tutorial_runs/opt_parser_raw/example_survey/ollama_smollm2:360m-instruct-fp16_SingleTurn
----<>----
{'temperature': 0}
--<chat model>-- model='smollm2:360m-instruct-fp16' temperature=0.0


2026-05-03 03:47:56.986 | CRITICAL | psychscanner.session_tunnel.session_tunnel:create_tunnel:152 - BEGIN


TOTAL RUNS: 1	RESUME IDX: None
----<>---- task running


6it [00:04,  1.32it/s]
2026-05-03 03:48:01.611 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-05-03 03:48:01.614 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


--- pred_resp with parser_raw=True ---
AIMessage(content='{\n    "Vividness": 1\n}', additional_kwargs={}, response_metadata={'model': 'smollm2:360m-instruct-fp16', 'created_at': '2026-05-03T07:47:57.951962Z', 'done': True, 'done_reason': 'stop', 'total_duration': 889618866, 'load_duration': 90092596, 'prompt_eval_count': 106, 'prompt_eval_duration': 201503701, 'eval_count': 12, 'eval_duration': 490280054, 'logprobs': None, 'model_name': 'smollm2:360m-instruct-fp16', 'model_provider': 'ollama'}, id='lc_run--019decce-ec04-79f2-b5a9-690f452a0884-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 106, 'output_tokens': 12, 'total_tokens': 118})


## 8. `parser_config` — controlling the structured-output method

`parser_config` is forwarded as `**kwargs` to `model.with_structured_output(...)`. The default is `{"method": "json_schema"}`. Other LangChain-supported values include `"function_calling"` (tool-call based) and `"json_mode"` — availability depends on the provider.

In [18]:
card_cfg = ExpCardInit()
card_cfg.proj_dir      = PROJECT_DIR/"8"
card_cfg.projectname   = "opt_parser_config"
card_cfg.model         = MODEL_NAME
card_cfg.family        = MODEL_FAMILY
card_cfg.parameters    = {"temperature": 0}
card_cfg.task_file     = task_file
card_cfg.parser        = DefaultLiteralAgree
card_cfg.parser_config = {"method": "json_schema"}   # try "function_calling" if your model supports it
card_cfg.cogtype       = "no"
card_cfg.nsim          = 1
card_cfg.tunnel_status = "0"

exp = ExpCard(card_cfg)
scanner = ScannerModel(expcard=exp)
results_cfg = scanner.run()

pprint(results_cfg[0][0]["pred_resp"])

----<PROJECT AND DATA ROOT DIRECTORY>----
	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_parser_tutorial_runs/8
	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_parser_tutorial_runs/8/opt_parser_config/example_survey/ollama_smollm2:360m-instruct-fp16_SingleTurn
----<>----
{'temperature': 0}
--<chat model>-- model='smollm2:360m-instruct-fp16' temperature=0.0


2026-05-03 03:49:11.651 | CRITICAL | psychscanner.session_tunnel.session_tunnel:create_tunnel:152 - BEGIN


TOTAL RUNS: 1	RESUME IDX: None
----<>---- task running


6it [00:03,  1.50it/s]
2026-05-03 03:49:15.666 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-05-03 03:49:15.670 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


AIMessage(content="{'rating': 5}", additional_kwargs={}, response_metadata={}, id='1ba758c3-196b-4cc7-b16e-b16471d094b5', tool_calls=[], invalid_tool_calls=[])


## 9. Summary — quick reference

```python
# No parsing
card.parser = "0"

# Read parser name from task JSON's "parser" field
card.parser = "1"

# Pass a Pydantic class directly (recommended)
card.parser = MyParserClass

# Hardcoded RM dynamic routing (encode vs test trcode)
card.parser = "dynamic"

# Modifiers
card.parser_raw    = True                          # keep raw output
card.parser_config = {"method": "json_schema"}     # passed to with_structured_output
```

### Gotchas

- `mock-llm` does **not** support structured output — only mode `"0"` works with it.
- Mode `"1"` uses `eval()` on the class name string. The class must be imported in `scanner_cards.py` (which does `from psychscanner.datasets.prompts.parser import *`).
- Mode `"dynamic"` is hardcoded to `Response_part_1_rm` / `Response_part_2_rm` and dispatches on the substring `"test"` in `trcode`. To use it for any other task you'd need to edit `single_turn_convo.py`.
- `trial_parsers` exists as a field on `ExpCardInit`/`AgentConfig` but is not currently wired into execution.